**Schema e definição das chaves de negócio**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS erp_lakehouse.silver")

primary_keys = {
    "customer": ["c_custkey"],
    "orders": ["o_orderkey"],
    "lineitem": ["l_orderkey", "l_linenumber"],  # chave composta
    "part": ["p_partkey"],
    "supplier": ["s_suppkey"],
}

**Função genérica de transformação Silver**

In [0]:
def make_silver(tabela: str, keys: list):
    df = spark.table(f"erp_lakehouse.bronze.{tabela}")

    # Remove colunas técnicas de ruído que não vão pra Silver
    df = df.drop("_rescued_data")

    # Deduplica pela chave de negócio, mantendo o registro mais recente
    window = Window.partitionBy(*keys).orderBy(F.col("_ingested_at").desc())
    df = (df.withColumn("_rn", F.row_number().over(window))
            .filter("_rn = 1")
            .drop("_rn"))

    # Regra de qualidade: chave de negócio nunca pode ser nula
    for k in keys:
        df = df.filter(F.col(k).isNotNull())

    df = df.withColumn("_processed_at", F.current_timestamp())

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"erp_lakehouse.silver.{tabela}"))

    print(f"✅ silver.{tabela}: {df.count()} linhas")

**Executar para todas as tabelas**

In [0]:
for tabela, keys in primary_keys.items():
    make_silver(tabela, keys)

**Adicionar constraints de negócio**

In [0]:
def add_constraint_if_not_exists(table: str, constraint_name: str, condition: str):
    try:
        spark.sql(f"""
            ALTER TABLE {table}
            ADD CONSTRAINT {constraint_name} CHECK ({condition})
        """)
        print(f"✅ Constraint '{constraint_name}' criada em {table}")
    except Exception as e:
        if "already exists" in str(e):
            print(f"⏭️  Constraint '{constraint_name}' já existe em {table}, pulando")
        else:
            raise  # se for outro tipo de erro, não engole silenciosamente

add_constraint_if_not_exists(
    "erp_lakehouse.silver.orders", "valid_totalprice", "o_totalprice >= 0"
)
add_constraint_if_not_exists(
    "erp_lakehouse.silver.lineitem", "valid_quantity", "l_quantity > 0"
)